###ZORDER IN DATABRICKS (DELTA LAKE)
####ZORDER:
- ZORDER is an optional feature used with OPTIMIZE to colocate related data physically in the same set of files.
- It improves query performance for range or equality filters on the specified columns.
####USAGE:
- OPTIMIZE table_name [WHERE predicate] ZORDER BY (col1, col2, ...)
- Reduces file scan for queries filtering on ZORDER columns.
- Works best for columns used frequently in WHERE clauses.
- Only reorganizes existing data; does not add or remove rows.
####EXAMPLE USE CASE:
- Periodically optimize large Delta tables with frequent writes/updates.
- Use ZORDER on high-selectivity columns to improve read performance.

In [0]:
%sql
-- Drop table if it exists
-- Catalog: new_catalog
-- Schema: default_schema
-- Table: customer_txn
-- This ensures the table is removed safely without error if it does not exist

DROP TABLE IF EXISTS new_catalog.default_schema.customer_txn;

In [0]:
%sql
-- Step 1 – Create the Delta table

-- Create table only if it does not already exist
-- Catalog: new_catalog
-- Schema: default_schema
-- Table: customer_txn
-- Purpose: Store customer transaction details in Delta format

CREATE TABLE IF NOT EXISTS new_catalog.default_schema.customer_txn (
    txn_id INT,              -- Unique transaction identifier
    customer_id INT,         -- Customer identifier
    region STRING,           -- Geographic region of the customer
    txn_amount DOUBLE,       -- Transaction amount
    txn_type STRING,         -- Type of transaction (e.g., purchase, refund)
    transaction_date DATE    -- Date of the transaction
)
USING DELTA;                 -- Delta Lake format for ACID compliance and performance

In [0]:
%sql
-- Step 2 – Insert multiple small batches
-- Each insert writes a few small Parquet files into Delta storage.
-- This simulates fragmented data loads that will later benefit from OPTIMIZE.

-- Batch 1: Initial customer transactions
INSERT INTO new_catalog.default_schema.customer_txn VALUES
 (1, 1001, 'North', 250.00, 'Online', '2025-10-01'),
 (2, 1002, 'South', 400.00, 'Offline', '2025-10-02'),
 (3, 1003, 'West', 600.00, 'Online', '2025-10-03');

-- Batch 2: Additional transactions from different regions
INSERT INTO new_catalog.default_schema.customer_txn VALUES
 (4, 1001, 'North', 300.00, 'Offline', '2025-10-01'),
 (5, 1004, 'East', 750.00, 'Online', '2025-10-02'),
 (6, 1005, 'South', 180.00, 'Online', '2025-10-03');

-- Batch 3: More transactions including repeat customers
INSERT INTO new_catalog.default_schema.customer_txn VALUES
 (7, 1001, 'North', 270.00, 'Online', '2025-10-01'),
 (8, 1003, 'West', 500.00, 'Offline', '2025-10-02'),
 (9, 1002, 'South', 900.00, 'Online', '2025-10-03');

In [0]:
%sql
/*
region=North
    - part-0
    - part-1
    - part-2
region=South
    - part-0
    - part-1
    - part-2
region=West
    - part-0
    - part-1
region=East
    - part-0

select * from new_catalog.default_schema.customer_txn where region='North';

optimize new_catalog.default_schema.customer_txn;
region=North
    - part-0
    - part-1
    - part-2
    - part-3 - after optimize
region=South
    - part-0
    - part-1
    - part-2
    - part-3 - after optimize
region=West
    - part-0
    - part-1
    - part-2 - after optimize
region=East
    - part-0
    - part-1 - after optimize


optimize inceptez_catalog.inputdb.customer_txn zorder by transaction_date;

optimize new_catalog.default_schema.customer_txn;
region=North
    - part-0
    - part-1
    - part-2
    - part-3 - optimize & sort the data rows in transaction_date
region=South
    - part-0
    - part-1
    - part-2
    - part-3 - optimize & sort the data rows in transaction_date
region=West
    - part-0
    - part-1
    - part-2 - optimize & sort the data rows in transaction_date
region=East
    - part-0
    - part-1 - optimize & sort the data rows in transaction_date
*/



In [0]:
%sql
-- Step 3: Inspect table metadata
-- DESCRIBE DETAIL shows catalog, schema, table type, provider, location,
-- row count, size in bytes, and other properties.
DESCRIBE DETAIL new_catalog.default_schema.customer_txn;


In [0]:
%sql

-- Step 4: Optimize table storage
-- OPTIMIZE compacts many small Parquet files into fewer large files.
-- This improves query performance and reduces overhead.
-- Recommended after multiple small batch inserts.
OPTIMIZE new_catalog.default_schema.customer_txn ZORDER BY (transaction_date);

In [0]:
%sql
-- Step 5: Inspect table metadata
-- DESCRIBE DETAIL shows catalog, schema, table type, provider, location,
-- row count, size in bytes, and other properties.
DESCRIBE DETAIL new_catalog.default_schema.customer_txn;
